# MOP × DP2 — paso 1: snapshot y enriquecimiento MOP

Este notebook descarga el catálogo completo de targets disponible en MOP y consulta, para cada target, la página que contiene sus parámetros y su fotometría.

La corrida empieza limitada a 200 targets. Para procesar todo el snapshot, cambiar `TARGET_LIMIT = None` en la celda de parámetros. Los resultados de cada target se guardan en caches separados, de modo que una corrida interrumpida puede reanudarse.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import sys

import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from IPython.display import display
from datetime import datetime, timezone

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

# El notebook funciona tanto abierto desde la raíz del repo como desde esta carpeta.
repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((path for path in repo_candidates if (path / 'configs' / 'mop_dp2_all.toml').exists()), Path('/home/karennowo/target_selection'))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import tomllib
except ModuleNotFoundError:  # pragma: no cover
    import tomli as tomllib

from mop_api import MOPClient
from target_selection.sources.mop import apply_authoritative_coordinates

CONFIG_PATH = REPO_ROOT / 'configs' / 'mop_dp2_all.toml'
config = tomllib.loads(CONFIG_PATH.read_text(encoding='utf-8'))
experiment = config['experiment']
paths_config = config['paths']

OUTPUT_DIR = REPO_ROOT / paths_config['output_dir']
RAW_DIR = REPO_ROOT / paths_config['raw_dir']
CACHE_DIR = REPO_ROOT / paths_config['cache_dir']
TABLES_DIR = REPO_ROOT / paths_config['tables_dir']
MOP_EVENT_CACHE = CACHE_DIR / 'mop_event_cache'
MOP_PHOTOMETRY_CACHE = CACHE_DIR / 'mop_photometry'
for directory in (OUTPUT_DIR, RAW_DIR, CACHE_DIR, TABLES_DIR, MOP_EVENT_CACHE, MOP_PHOTOMETRY_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

# Preview para validar el flujo. Para enriquecer los 6555 eventos usar None.
TARGET_LIMIT = None
RUN_DOWNLOAD = False
RUN_ENRICHMENT = True
MAX_WORKERS = int(experiment.get('max_workers', 2))
REFRESH = bool(experiment.get('refresh', False))

print('REPO_ROOT:', REPO_ROOT)
print('TARGET_LIMIT:', TARGET_LIMIT)
print('MAX_WORKERS:', MAX_WORKERS)


## 1. Descargar el snapshot bruto de MOP

La descarga se hace directamente sobre el listado paginado `/targets/`. La API guarda cada página y luego concatena todo en un único snapshot. No se usa `/targets/export/`, porque ese endpoint puede tardar o devolver 504 sin aportar información adicional.


In [ ]:
def download_mop_targets():
    mop = MOPClient(timeout=180, retries=4)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    raw_path = RAW_DIR / f'mop_targets_{stamp}.csv'
    raw = mop.download_all_targets(
        output_dir=RAW_DIR,
        save_path=raw_path,
        page_delay=0.25,
    )
    metadata = dict(raw.attrs.get('mop_download', {}))
    metadata['raw_path'] = str(raw_path)
    metadata['sha256'] = sha256_bytes(raw_path.read_bytes())
    metadata['columns'] = [str(column) for column in raw.columns]
    return mop, raw, raw_path, metadata

if RUN_DOWNLOAD:
    mop, mop_raw, raw_path, snapshot_metadata = download_mop_targets()
else:
    candidates = sorted(RAW_DIR.glob('mop_targets_*.csv'))
    if not candidates:
        raise FileNotFoundError('No hay snapshot MOP local. Activar RUN_DOWNLOAD.')
    mop = MOPClient(timeout=180, retries=4)
    raw_path = candidates[-1]
    mop_raw = pd.read_csv(raw_path)
    snapshot_metadata = {
        'downloaded_at_utc': raw_path.stem.removeprefix('mop_targets_'),
        'raw_path': str(raw_path),
        'method': 'local_snapshot',
        'n_rows': int(len(mop_raw)),
        'columns': [str(column) for column in mop_raw.columns],
        'sha256': sha256_bytes(raw_path.read_bytes()),
    }

print(f"Snapshot: {raw_path}")
print(f"Rows: {len(mop_raw):,}")
print("Columns:", list(mop_raw.columns))
display(mop_raw.head())


In [ ]:
if snapshot_metadata.get('method') == 'paginated_html':
    page_dir = Path(snapshot_metadata['page_dir'])
    page_files = sorted(page_dir.glob('mop_targets_page_*.csv'))
    page_numbers = [int(path.stem.rsplit('_', 1)[-1]) for path in page_files]
    target_column = 'Target' if 'Target' in mop_raw.columns else 'Name'
    page_targets = pd.concat([pd.read_csv(path)[target_column] for path in page_files], ignore_index=True)
    duplicate_targets = page_targets.astype(str).duplicated().sum()
    expected_pages = snapshot_metadata.get('page_count')
    print(f'Páginas guardadas: {len(page_files)} / {expected_pages}')
    print(f'Filas de páginas: {len(page_targets):,}')
    print(f'Targets únicos: {page_targets.astype(str).nunique():,}')
    print(f'Duplicados: {duplicate_targets:,}')
    print(f'Primera/última página: {page_numbers[0]} / {page_numbers[-1]}')
    if page_numbers != list(range(1, len(page_files) + 1)) or duplicate_targets:
        raise ValueError('La descarga tiene páginas faltantes o targets duplicados.')
    if expected_pages is not None and len(page_files) != int(expected_pages):
        raise ValueError('La cantidad de páginas guardadas no coincide con el manifiesto.')
else:
    print('Snapshot local: se omite la auditoría de páginas.')


## 2. Normalizar nombres y coordenadas

Se conserva una copia de todas las columnas MOP y se agregan las columnas canónicas `target_id`, `Target`, `RA_deg` y `Dec_deg`. Las coordenadas numéricas se interpretan como grados; las coordenadas sexagesimales se convierten con Astropy.

In [ ]:
def normalized_label(value) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(value).casefold())

def find_column(frame, candidates, required=True):
    by_label = {normalized_label(column): column for column in frame.columns}
    for candidate in candidates:
        if normalized_label(candidate) in by_label:
            return by_label[normalized_label(candidate)]
    if required:
        raise KeyError(f'No se encontró una columna entre {candidates}; columnas: {list(frame.columns)}')
    return None

def parse_coordinate_pair(ra_value, dec_value):
    numeric_ra = pd.to_numeric(pd.Series([ra_value]), errors='coerce').iloc[0]
    numeric_dec = pd.to_numeric(pd.Series([dec_value]), errors='coerce').iloc[0]
    if pd.notna(numeric_ra) and pd.notna(numeric_dec):
        if 0 <= float(numeric_ra) < 360 and -90 <= float(numeric_dec) <= 90:
            return float(numeric_ra), float(numeric_dec)
    coordinate = SkyCoord(str(ra_value), str(dec_value), unit=(u.hourangle, u.deg), frame='icrs')
    return float(coordinate.ra.deg), float(coordinate.dec.deg)

def normalize_mop_targets(raw):
    name_column = find_column(raw, ('Target', 'Name', 'target_name', 'object'))
    ra_column = find_column(raw, ('RA_deg', 'ra_deg', 'Right Ascension', 'RA', 'ra'))
    dec_column = find_column(raw, ('Dec_deg', 'dec_deg', 'Declination', 'Dec', 'dec'))
    type_column = find_column(raw, ('Target Type', 'target_type', 'type'), required=False)

    result = raw.copy()
    target_ids = raw[name_column].astype(str).str.strip()
    if 'target_id' in result.columns:
        result['target_id'] = target_ids
    else:
        result.insert(0, 'target_id', target_ids)
    if 'Target' in result.columns:
        result['Target'] = result['target_id']
    else:
        result.insert(1, 'Target', result['target_id'])
    coordinates = [parse_coordinate_pair(ra, dec) for ra, dec in zip(raw[ra_column], raw[dec_column], strict=True)]
    result['RA_deg'] = [pair[0] for pair in coordinates]
    result['Dec_deg'] = [pair[1] for pair in coordinates]
    result['target_type'] = raw[type_column].astype(str) if type_column else ''
    result['mop_snapshot_source_column'] = name_column
    return result

targets = normalize_mop_targets(mop_raw)
duplicate_mask = targets['target_id'].duplicated(keep=False)
invalid_coordinates = (
    ~targets['RA_deg'].between(0, 360, inclusive='left')
    | ~targets['Dec_deg'].between(-90, 90, inclusive='both')
)

print(f'Targets normalizados: {len(targets):,}')
print(f'Duplicados: {int(duplicate_mask.sum()):,}')
print(f'Coordenadas inválidas: {int(invalid_coordinates.sum()):,}')
if duplicate_mask.any():
    display(targets.loc[duplicate_mask, ['target_id', 'RA_deg', 'Dec_deg']].head(20))
if invalid_coordinates.any():
    display(targets.loc[invalid_coordinates, ['target_id', 'RA_deg', 'Dec_deg']].head(20))

if duplicate_mask.any() or invalid_coordinates.any():
    raise ValueError('El snapshot necesita revisión antes de enriquecerlo.')

if TARGET_LIMIT is not None:
    targets = targets.head(int(TARGET_LIMIT)).copy()

snapshot_path = TABLES_DIR / 'mop_targets_snapshot.csv'
targets.to_csv(snapshot_path, index=False)
print(f'Snapshot normalizado guardado en: {snapshot_path}')
display(targets[['target_id', 'Target', 'RA_deg', 'Dec_deg', 'target_type']].head())


## 3. Descargar parámetros y fotometría de cada página MOP

Primero se consulta la página individual de cada target y se guardan sus parámetros, estado (`Alive`), `Classification` y `Category`. La fotometría se descarga solamente si el target está activo y su clasificación o categoría contiene `Microlensing`. Los demás eventos quedan registrados con `mop_photometry_status = skipped`.


In [ ]:
def should_download_mop_photometry(row):
    alive = str(row.get('mop_alive', '')).strip().casefold() in {'true', '1', 'yes'}
    classification = str(row.get('mop_classification', '')).casefold()
    category = str(row.get('mop_category', '')).casefold()
    is_microlensing = 'microlens' in classification or 'microlens' in category
    return alive and is_microlensing

if RUN_ENRICHMENT:
    columns = ['Target', 'RA_deg', 'Dec_deg']
    if 'mop_link' in targets.columns:
        columns.append('mop_link')
    mop_targets = targets[columns].copy()
    enriched = mop.enrich_microlensing_parameters(
        mop_targets,
        target_column='Target',
        prefix='mop_',
        errors='ignore',
        max_workers=MAX_WORKERS,
        cache_dir=MOP_EVENT_CACHE,
        photometry_dir=MOP_PHOTOMETRY_CACHE,
        photometry_selector=should_download_mop_photometry,
        refresh=REFRESH,
    )
    enriched = apply_authoritative_coordinates(enriched)
else:
    enriched_path = TABLES_DIR / 'mop_targets_enriched.csv'
    if not enriched_path.exists():
        raise FileNotFoundError('No existe el catálogo enriquecido. Activar RUN_ENRICHMENT.')
    enriched = pd.read_csv(enriched_path)

enriched_path = TABLES_DIR / 'mop_targets_enriched.csv'
enriched.to_csv(enriched_path, index=False)
print(f'Catálogo enriquecido: {enriched_path}')
print(f'Filas: {len(enriched):,}')
if 'mop_photometry_status' in enriched:
    print('Estado de fotometría:')
    display(enriched['mop_photometry_status'].value_counts(dropna=False).rename_axis('status').to_frame('n'))
display(enriched.head())


## 4. Resumen de estados y fotometría cacheada

Estos archivos son los insumos del siguiente notebook. El índice de fotometría evita tener que abrir todos los CSV para saber qué targets tienen datos.

In [ ]:
def safe_cache_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('._') or 'unknown'

status = enriched[['Target']].copy()
for column in ('mop_link', 'mop_alive', 'mop_classification', 'mop_category', 'mop_parameters_status', 'mop_parameters_error', 'mop_photometry_status', 'mop_photometry_reason', 'mop_photometry_error', 'mop_ra_deg', 'mop_dec_deg'):
    status[column] = enriched[column] if column in enriched else pd.NA
status['has_mop_parameters'] = status['mop_parameters_status'].astype(str).str.casefold().eq('available')
status['has_mop_photometry'] = status['mop_photometry_status'].astype(str).str.casefold().eq('available')
status['event_cache_path'] = status['Target'].map(lambda value: str(MOP_EVENT_CACHE / f'{safe_cache_name(value)}.json'))
status['photometry_cache_path'] = status['Target'].map(lambda value: str(MOP_PHOTOMETRY_CACHE / f'{safe_cache_name(value)}.csv'))

index_rows = []
for target_name in enriched['Target'].astype(str):
    path = MOP_PHOTOMETRY_CACHE / f'{safe_cache_name(target_name)}.csv'
    record = {'Target': target_name, 'photometry_path': str(path), 'n_points': 0, 'filters': '', 'first_timestamp': '', 'last_timestamp': '', 'photometry_read_status': 'missing'}
    if path.exists():
        try:
            data = pd.read_csv(path)
            record['n_points'] = int(len(data))
            if 'Filter' in data:
                record['filters'] = ','.join(sorted(data['Filter'].dropna().astype(str).unique()))
            if 'Timestamp' in data and len(data):
                timestamps = pd.to_datetime(data['Timestamp'], errors='coerce', utc=True).dropna()
                if len(timestamps):
                    record['first_timestamp'] = timestamps.min().isoformat()
                    record['last_timestamp'] = timestamps.max().isoformat()
            record['photometry_read_status'] = 'ok'
        except Exception as error:
            record['photometry_read_status'] = f'{type(error).__name__}: {error}'
    index_rows.append(record)

photometry_index = pd.DataFrame(index_rows)
status = status.merge(photometry_index, on='Target', how='left')
status_path = TABLES_DIR / 'mop_event_status.csv'
index_path = TABLES_DIR / 'mop_photometry_index.csv'
status.to_csv(status_path, index=False)
photometry_index.to_csv(index_path, index=False)

manifest = {
    **snapshot_metadata,
    'target_limit': TARGET_LIMIT,
    'n_targets_analyzed': int(len(enriched)),
    'n_targets_with_parameters': int(status['has_mop_parameters'].sum()),
    'n_targets_with_photometry': int(status['has_mop_photometry'].sum()),
    'n_targets_photometry_skipped': int(status['mop_photometry_status'].astype(str).eq('skipped').sum()),
    'n_targets_classified_microlensing': int((status['mop_classification'].astype(str).str.contains('microlens', case=False, na=False) | status['mop_category'].astype(str).str.contains('microlens', case=False, na=False)).sum()),
    'n_photometry_points': int(photometry_index['n_points'].sum()),
    'max_workers': MAX_WORKERS,
    'refresh': REFRESH,
    'event_cache_dir': str(MOP_EVENT_CACHE),
    'photometry_cache_dir': str(MOP_PHOTOMETRY_CACHE),
}
manifest_path = OUTPUT_DIR / 'mop_snapshot_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

print(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
print('Estados:', status_path)
print('Índice de fotometría:', index_path)
display(status[['Target', 'mop_parameters_status', 'mop_photometry_status', 'n_points', 'filters']].head(20))


## Productos generados

- `raw/mop_targets_<timestamp>.csv`: respuesta original de MOP.
- `tables/mop_targets_snapshot.csv`: snapshot normalizado usado en esta corrida.
- `tables/mop_targets_enriched.csv`: parámetros MOP y coordenadas de las páginas.
- `tables/mop_event_status.csv`: estado de descarga por target.
- `tables/mop_photometry_index.csv`: cantidad, filtros y rango temporal de fotometría.
- `cache/mop_event_cache/`: un JSON por target.
- `cache/mop_photometry/`: un CSV por target.
- `mop_snapshot_manifest.json`: provenance y conteos de la corrida.